In [12]:
import torch
import numpy 
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.transforms import  v2



In [13]:
transform = v2.Compose([v2.ToImage(),v2.ToDtype(dtype=torch.float32,scale=True)])

In [14]:
training_data = datasets.CIFAR10(
    root="../data",
    download=True,
    train=True,
    transform=transform
)

test_data = datasets.CIFAR10(
    root="../data",
    download=True,
    train=False,
    transform=transform   
)

In [19]:
from torch.utils.data import Subset

train_data = Subset(training_data, range(49000))
val_data = Subset(training_data, range(49000, 50000))

In [20]:
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=64,
    shuffle=False
)

In [21]:
test_data = Subset(test_data, range(1000))

In [23]:
images,labels = next(iter(train_loader))
print(images[0].shape) 
print(images[0].shape)

torch.Size([3, 32, 32])
torch.Size([3, 32, 32])


In [22]:
test_loader  = DataLoader(dataset=test_data,batch_size=64)

In [32]:
class Two_Layer_net:
    def __init__(self,input_size,hidden_layer_1_size,output_layer_size=10,batch_size=64,weight_scale=1e-3,reg=0):
        
        self.input_dim=input_size
        self.hidden_dim=hidden_layer_1_size
        self.num_classes=output_layer_size
        self.batch_size=batch_size

        #weights initialization and regularization

        self.params = {}
        self.reg = reg 

        self.params = {
          'W1': np.random.randn(self.input_dim, self.hidden_dim) * weight_scale,
          'b1': np.zeros(self.hidden_dim),
          'W2': np.random.randn(self.hidden_dim, self.num_classes) * weight_scale,
          'b2': np.zeros(self.num_classes)
        }

    def softmax(self,x):
        "x is 2 dim np array" 
        x = x - np.max(x, axis=1, keepdims=True)
        exp_scores = np.exp(x)
        probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        return probabilities

    def ReLU(self, x):
        return np.maximum(0, x)

    def forward(self,x):
        """
        x is an batch input of dimension (batch,size,3,32,32)
        architecture => affine - relu - affine -softmax(scores) 
        
        """
        #flatten the image 
        x = x.reshape(x.shape[0], -1)
        #pass it to the first layer
        self.layer1_output = np.matmul(x,self.params["W1"]) + self.params["b1"]
        #pass it through relu 
        self.relu_ouptut = self.ReLU(self.layer1_output) 
        #pass the relu output to the next layer 
        self.layer2_output = np.matmul(self.relu_ouptut,self.params["W2"]) + self.params["b2"]
        #pass it to softmax to get the scores [0,1] 

        #layer_2 output is our score from forward pass
        return self.layer2_output

    def loss(self, x, y):
        scores = self.forward(x)
        probabilities = self.softmax(scores)
    
        # correct-class probabilities
        correct_probs = probabilities[np.arange(x.shape[0]), y]
    
        # average cross-entropy loss
        data_loss = -np.mean(np.log(correct_probs))
    
        # L2 regularization
        reg_loss = 0.5 * self.reg * (
            np.sum(self.params["W1"] ** 2)
            + np.sum(self.params["W2"] ** 2)
        )
    
        return data_loss + reg_loss
        